# Lecture 3.4 — Web Tools: WebSearch & WebFetch

**Course:** Build Production AI Agents with the Claude Agent SDK  
**Section:** 3 — Built-In Tools Deep Dive

---

In Lectures 3.1–3.3 we built agents that read files, ran shell commands, and searched the local filesystem.

In this lecture we cross a new boundary entirely. By the end of this notebook, your agent will be able to:

- **Search the live web** using `WebSearch` — discover current information and synthesise it into a coherent answer
- **Fetch full page content** using `WebFetch` — retrieve and parse any URL into readable text
- **Combine both tools** into a complete research workflow: discover, read, synthesise

This notebook is **self-contained** — it installs the SDK and loads the API key from Colab Secrets.  
No other lecture notebook is required.

In [1]:
# Install the SDK

# We pin the Claude Agent SDK to a specific version to ensure all examples
# in this notebook run exactly as shown in the course. If you encounter any
# issues or want to experiment with newer features, you can install the latest
# version by removing the version pin (replace 'claude-agent-sdk==0.2.93' with just
# 'claude-agent-sdk'). Note that newer versions may behave differently from
# what is demonstrated in the videos. We will update the notebooks periodically
# to keep up with new releases.

!pip install claude-agent-sdk==0.2.93 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 MB 10.1 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Model Configuration

The cell below sets `MODEL_NAME` — the Claude model used for every agent in this notebook.

You can change this to any model you have access to.  
For the latest available models, visit: https://platform.claude.com/docs/en/about-claude/models/overview

Every `ClaudeAgentOptions` call in this notebook passes `model=MODEL_NAME`,  
so you only need to change it in one place.

In [3]:
# Model configuration
# Change this to use a different Claude model
# For the latest available models visit:
# https://platform.claude.com/docs/en/about-claude/models/overview
MODEL_NAME = "claude-haiku-4-5"

## Why Web Tools Matter — Breaking Beyond the Local Filesystem

In Lectures 3.1–3.3, every tool we used operated on **local data** — files and folders on your own machine.

Claude also has a **training cutoff**: information that came into existence after training simply is not in its memory.

**Web tools break both of those limits.** `WebSearch` and `WebFetch` give the agent access to live, current information from the internet. This is especially powerful for:

- Research tasks — synthesising findings across multiple sources
- Finding the latest documentation — no stale training data
- Checking current information — versions, prices, news, anything that changes
- Any task that requires knowledge beyond Claude's training cutoff

**Colab has internet access by default** — both tools work out of the box with no additional setup.

> ⚠️ **Live output will vary each time you run a cell.** This is not a bug — it is the nature of real-time web data.  
> Evaluate your output by **structure and quality**, not by matching specific words or version numbers.

## The WebSearch Tool

`WebSearch` performs a **live web search** and returns relevant results — titles, snippets, and URLs.

Claude does not just hand back a list of links. It reads the results and **synthesises them into a coherent, structured answer**.

**When to use `WebSearch`:**
- You need to find current information on a topic
- You are researching something and want synthesised findings from multiple sources
- You need to discover relevant URLs to fetch later with `WebFetch`

Output will vary based on live search results — this is expected and normal.

In [4]:
from claude_agent_sdk import query, ClaudeAgentOptions

async for message in query(
    prompt="""
    Search the web for Claude Agent SDK Python examples.
    Find and summarise:
    1. What the Claude Agent SDK is used for
    2. A simple Python example showing how to use it
    3. Where students can find more examples and documentation
    Present as a clean structured summary.
    """,
    options=ClaudeAgentOptions(
        allowed_tools=["WebSearch"],
        model=MODEL_NAME,
    ),
):
    if hasattr(message, "result"):
        print(message.result)

Perfect! Now I have comprehensive information. Let me create a structured summary for you:

---

# Claude Agent SDK Python: Complete Summary

## 1. What is the Claude Agent SDK Used For?

The **Claude Agent SDK** is a Python library that enables you to build **production AI agents** that autonomously perform tasks typically requiring human interaction. Key capabilities include:

- **File Operations**: Read, write, and edit files in your codebase
- **Command Execution**: Run terminal commands and scripts (including git operations)
- **Code Analysis**: Search codebases, find patterns, and understand code structure
- **Web Integration**: Search the web and fetch/parse web page content
- **Autonomous Workflows**: Build agents that work independently without manual intervention
- **Custom Tools & Hooks**: Define custom behaviors and lifecycle callbacks for specialized use cases
- **Subagents**: Delegate work to specialized agents that focus on specific subtasks
- **MCP Integration**: Connec

## The WebFetch Tool

`WebFetch` **retrieves the full content of a specific URL** and parses it into clean readable text that Claude can reason about.

**How it differs from `WebSearch`:**

| | `WebSearch` | `WebFetch` |
|---|---|---|
| **You provide** | A topic or query | A specific URL |
| **Returns** | Snippets from multiple sources | Full content of one page |
| **Best for** | Discovering information | Deep reading a known page |

**When to use `WebFetch`:**
- You already have a specific URL
- You need the complete content of that page — not just a search snippet
- You want to extract, summarise, or reason about the full page

No `requests` library, no `BeautifulSoup`, no HTML parsing code — the agent handles all of that for you.

In [5]:
async for message in query(
    prompt="""
    Fetch the page at https://pypi.org/project/claude-agent-sdk/
    and extract the following information:
    1. The current version of the SDK
    2. A one sentence description of what the SDK does
    3. The pip install command
    4. Any listed requirements or dependencies
    Present as a clean structured report.
    """,
    options=ClaudeAgentOptions(
        allowed_tools=["WebFetch"],
        model=MODEL_NAME,
    ),
):
    if hasattr(message, "result"):
        print(message.result)

Perfect! Here's the structured report for the Claude Agent SDK:

---

## **Claude Agent SDK - PyPI Information**

### **Current Version**
**0.2.93** (Released June 6, 2026)

### **Description**
Python SDK for Claude Agent that enables programmatic interaction with Claude Code through async queries and bidirectional conversations.

### **Installation Command**
```bash
pip install claude-agent-sdk
```

### **Requirements & Dependencies**

**Python Requirements:**
- Python 3.10 or higher (supports 3.10, 3.11, 3.12, 3.13)

**Included:**
- Claude Code CLI is automatically bundled with the package
- Optionally customizable via `ClaudeAgentOptions(cli_path="/path/to/claude")`

**Optional Extras:**
- `dev` - Development dependencies
- `examples` - Example code files
- `otel` - OpenTelemetry support

**License:** MIT License

---


## WebSearch + WebFetch Together — A Complete Research Workflow

The two tools work best as a pair:

1. **`WebSearch`** — discovers relevant URLs on a topic
2. **`WebFetch`** — reads the full content of the most relevant one

This is the same **search-then-retrieve** pattern we saw with `Glob` and `Grep` in Lecture 3.3 — find first, read second — but now pointed at the entire internet.

When you allow both tools, the agent decides **autonomously**:
- Search for relevant URLs
- Evaluate the results
- Choose the most relevant page
- Fetch its full content
- Synthesise a coherent answer

All from a single prompt. You write the goal — the agent handles the workflow.

In [6]:
async for message in query(
    prompt="""
    Search the web for Claude Agent SDK Python quickstart examples.
    Find the most relevant page from the official documentation.
    Fetch that page and summarise:
    1. The key steps to get started with the SDK
    2. The first code example shown
    3. What tools are demonstrated
    Present as a clean getting-started summary.
    """,
    options=ClaudeAgentOptions(
        allowed_tools=["WebSearch", "WebFetch"],
        model=MODEL_NAME,
    ),
):
    if hasattr(message, "result"):
        print(message.result)

Perfect! I now have the official documentation. Here's a clean getting-started summary:

---

## Claude Agent SDK Python Quickstart Summary

### 1. Key Steps to Get Started

1. **Install the SDK** – Run `pip install claude-agent-sdk` (requires Python 3.10+)
2. **Set your API key** – Export `ANTHROPIC_API_KEY=your-api-key` from the Claude Platform Console
3. **Create your first agent** – Use the `query()` function with `ClaudeAgentOptions` to define allowed tools
4. **Stream results** – Use `async for` to process agent messages as they arrive

### 2. First Code Example

The simplest example from the official docs—listing files in your directory:

```python
import asyncio
from claude_agent_sdk import query, ClaudeAgentOptions

async def main():
    async for message in query(
        prompt="What files are in this directory?",
        options=ClaudeAgentOptions(allowed_tools=["Bash", "Glob"]),
    ):
        if hasattr(message, "result"):
            print(message.result)

asyncio.run(ma

## WebSearch vs WebFetch — When to Use Which

| | `WebSearch` | `WebFetch` |
|---|---|---|
| **You have** | A topic or question | A specific URL |
| **Returns** | Snippets from multiple sources | Full content of one page |
| **Best for** | Discovering information | Deep reading a known page |
| **Output** | Synthesised from multiple results | Parsed from one URL |

**Use both together when:** you want to research a topic and then deep-dive into the most relevant source — `WebSearch` finds it, `WebFetch` reads it fully.

> ⚠️ **Both tools require internet access.** They will not work in offline environments.  
> In Colab, internet access is enabled by default — both tools work out of the box.

## Summary — What We Covered in Lecture 3.4

In this lecture, the agent went live on the internet for the first time.

✅ **`WebSearch`** — live web search synthesised into a coherent answer. Used when you need to discover information and don't have a specific URL yet.

✅ **`WebFetch`** — full content of a specific URL, parsed and extracted. Used when you already have a URL and need everything on that page. No HTML parsing code required.

✅ **Combined workflow** — `WebSearch` discovers, `WebFetch` reads, Claude synthesises. One prompt, two tools, a complete autonomous research workflow.

✅ **Least-privilege tool access** — each demo allowed only the tool it needed. This discipline of minimal tool permissions becomes the focus of Section 4.

---

**Up next — Lecture 3.5: User Interaction with `AskUserQuestion`**

Every agent we have built so far runs autonomously from start to finish. But sometimes an agent needs to pause and ask a question — for clarification, a decision, or missing information it cannot find on its own.

In Lecture 3.5, we introduce `AskUserQuestion`: the tool that lets an agent present a question with multiple-choice options, wait for the human's answer, and continue based on that input. The bridge between fully autonomous agents and human-in-the-loop workflows.